# Homework 1: Concrete Compressive Strength Regression

该 Notebook 按作业 PDF 要求完成：

- 前 80% 训练，后 20% 测试
- 数据预处理
- 相关性分析
- PCA 主成分分析
- 线性回归
- Pytorch 神经网络回归
- MSE 评估与可视化

> 运行前请确认 `Concrete_Data_Yeh.csv` 与本 Notebook 位于同一目录。

In [ ]:
# -*- coding: utf-8 -*-
"""
混凝土抗压强度回归分析
根据作业 PDF 要求完成：
1. 使用前 80% 数据作为训练集，后 20% 数据作为测试集
2. 进行数据预处理
3. 进行特征分析（相关性分析 + PCA）
4. 使用线性回归与神经网络完成回归任务
5. 使用测试集进行评估，并绘图展示结果

运行环境建议：
- Python >= 3.9
- pandas, numpy, matplotlib, scikit-learn, torch

如果在 Jupyter Notebook 中运行，建议将本文件内容分单元格执行；
也可以直接运行该脚本。
"""

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
# 0. 全局设置

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

DATA_PATH = Path("Concrete_Data_Yeh.csv")  # 请确保 CSV 和当前 notebook/脚本位于同一目录
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 1. 读取数据

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"未找到数据文件：{DATA_PATH}\n"
        "请将 Concrete_Data_Yeh.csv 放在当前 notebook/脚本所在目录中，或修改 DATA_PATH。"
    )

# 原始列名
# ['cement', 'slag', 'flyash', 'water', 'superplasticizer',
#  'coarseaggregate', 'fineaggregate', 'age', 'csMPa']
df = pd.read_csv(DATA_PATH)

print("=" * 80)
print("1) 数据集基本信息")
print("=" * 80)
print("数据维度：", df.shape)
print("列名：", list(df.columns))
print("\n前 5 行数据：")
print(df.head())
print("\n缺失值统计：")
print(df.isnull().sum())
print("\n描述性统计：")
print(df.describe())

In [ ]:
# 2. 特征与标签划分

In [ ]:
feature_cols = [
    'cement', 'slag', 'flyash', 'water', 'superplasticizer',
    'coarseaggregate', 'fineaggregate', 'age'
]
target_col = 'csMPa'

X = df[feature_cols].copy()
y = df[target_col].copy()

In [ ]:
# 3. 按要求划分训练集 / 测试集
#    前 80% 训练，后 20% 测试

In [ ]:
train_size = int(len(df) * 0.8)

X_train = X.iloc[:train_size].reset_index(drop=True)
X_test = X.iloc[train_size:].reset_index(drop=True)
y_train = y.iloc[:train_size].reset_index(drop=True)
y_test = y.iloc[train_size:].reset_index(drop=True)

print("\n" + "=" * 80)
print("2) 数据划分")
print("=" * 80)
print(f"训练集大小: {X_train.shape[0]}")
print(f"测试集大小: {X_test.shape[0]}")

In [ ]:
# 4. 数据预处理
#    本数据无缺失值，这里仍保留标准流程

In [ ]:
# 若存在缺失值，可以使用以下方式：
# X_train = X_train.fillna(X_train.mean(numeric_only=True))
# X_test = X_test.fillna(X_train.mean(numeric_only=True))

# 标准化（仅用训练集拟合，再作用于测试集，避免数据泄露）
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 5. 相关性分析

In [ ]:
print("\n" + "=" * 80)
print("3) 相关性分析")
print("=" * 80)

corr_matrix = df.corr(numeric_only=True)
target_corr = corr_matrix[target_col].drop(target_col).sort_values(key=lambda s: s.abs(), ascending=False)

print("各特征与抗压强度的相关系数（按绝对值降序）：")
print(target_corr)

# 选择与目标最相关的前 5 个特征（你也可以改成前 4 或前 6）
selected_features = target_corr.index[:5].tolist()
print("\n选取的主要特征（Top 5）：", selected_features)

X_train_sel = X_train[selected_features].copy()
X_test_sel = X_test[selected_features].copy()

sel_scaler = StandardScaler()
X_train_sel_scaled = sel_scaler.fit_transform(X_train_sel)
X_test_sel_scaled = sel_scaler.transform(X_test_sel)

# 绘制特征与目标相关性柱状图
plt.figure(figsize=(10, 5))
plt.bar(target_corr.index, target_corr.values)
plt.xticks(rotation=45)
plt.title('各输入特征与混凝土抗压强度的相关系数')
plt.ylabel('Correlation with csMPa')
plt.tight_layout()
plt.show()

# 绘制相关系数矩阵热力图（matplotlib实现，避免额外依赖）
plt.figure(figsize=(9, 7))
plt.imshow(corr_matrix, aspect='auto')
plt.colorbar(label='Correlation')
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45, ha='right')
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
plt.title('相关系数矩阵热力图')
plt.tight_layout()
plt.show()

In [ ]:
# 6. PCA 主成分分析

In [ ]:
print("\n" + "=" * 80)
print("4) PCA 主成分分析")
print("=" * 80)

pca_full = PCA()
pca_full.fit(X_train_scaled)
explained_ratio = pca_full.explained_variance_ratio_
cum_explained_ratio = np.cumsum(explained_ratio)

print("各主成分解释方差比：")
for i, ratio in enumerate(explained_ratio, 1):
    print(f"PC{i}: {ratio:.4f}")

print("\n累计解释方差比：")
for i, ratio in enumerate(cum_explained_ratio, 1):
    print(f"前{i}个主成分: {ratio:.4f}")

# 选择累计解释方差 >= 95% 的主成分个数
n_components_95 = np.argmax(cum_explained_ratio >= 0.95) + 1
print(f"\n达到 95% 累计解释方差所需主成分数：{n_components_95}")

pca = PCA(n_components=n_components_95)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cum_explained_ratio) + 1), cum_explained_ratio, marker='o')
plt.axhline(y=0.95, linestyle='--')
plt.axvline(x=n_components_95, linestyle='--')
plt.xlabel('主成分个数')
plt.ylabel('累计解释方差比')
plt.title('PCA 累计解释方差曲线')
plt.tight_layout()
plt.show()

In [ ]:
# 7. 定义评估函数

In [ ]:
def evaluate_regression(y_true, y_pred, model_name="Model"):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"\n[{model_name}] 评估结果")
    print(f"MSE  = {mse:.4f}")
    print(f"RMSE = {rmse:.4f}")
    print(f"MAE  = {mae:.4f}")
    print(f"R²   = {r2:.4f}")

    return {
        "Model": model_name,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
    }


def plot_prediction(y_true, y_pred, model_name="Model"):
    # 真实值 vs 预测值折线图
    plt.figure(figsize=(12, 5))
    plt.plot(y_true.values, label='True')
    plt.plot(y_pred, label='Predicted')
    plt.title(f'{model_name}: 测试集真实值与预测值对比')
    plt.xlabel('样本编号')
    plt.ylabel('Concrete Strength (MPa)')
    plt.legend()
    plt.tight_layout()
    plt.show()

    # 散点图
    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, alpha=0.7)
    min_v = min(np.min(y_true), np.min(y_pred))
    max_v = max(np.max(y_true), np.max(y_pred))
    plt.plot([min_v, max_v], [min_v, max_v], linestyle='--')
    plt.xlabel('True Value')
    plt.ylabel('Predicted Value')
    plt.title(f'{model_name}: 真实值 vs 预测值')
    plt.tight_layout()
    plt.show()

In [ ]:
# 8. 线性回归模型

In [ ]:
print("\n" + "=" * 80)
print("5) 线性回归")
print("=" * 80)

# 8.1 使用全部特征
lr_all = LinearRegression()
lr_all.fit(X_train_scaled, y_train)
y_pred_lr_all = lr_all.predict(X_test_scaled)
res_lr_all = evaluate_regression(y_test, y_pred_lr_all, "Linear Regression (All Features)")
plot_prediction(y_test, y_pred_lr_all, "Linear Regression (All Features)")

# 8.2 使用相关性筛选后的特征
lr_sel = LinearRegression()
lr_sel.fit(X_train_sel_scaled, y_train)
y_pred_lr_sel = lr_sel.predict(X_test_sel_scaled)
res_lr_sel = evaluate_regression(y_test, y_pred_lr_sel, "Linear Regression (Selected Features)")
plot_prediction(y_test, y_pred_lr_sel, "Linear Regression (Selected Features)")

# 8.3 使用 PCA 后的特征
lr_pca = LinearRegression()
lr_pca.fit(X_train_pca, y_train)
y_pred_lr_pca = lr_pca.predict(X_test_pca)
res_lr_pca = evaluate_regression(y_test, y_pred_lr_pca, "Linear Regression (PCA Features)")
plot_prediction(y_test, y_pred_lr_pca, "Linear Regression (PCA Features)")

In [ ]:
# 9. 神经网络模型（Pytorch）

In [ ]:
print("\n" + "=" * 80)
print("6) 神经网络回归")
print("=" * 80)

class MLPRegressor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x)


# 转为 Tensor
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values.reshape(-1, 1), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values.reshape(-1, 1), dtype=torch.float32)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

model = MLPRegressor(input_dim=X_train.shape[1]).to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 300
train_losses = []

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(DEVICE)
        batch_y = batch_y.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * batch_X.size(0)

    epoch_loss /= len(train_loader.dataset)
    train_losses.append(epoch_loss)

    if (epoch + 1) % 50 == 0:
        print(f"Epoch [{epoch + 1:>3}/{num_epochs}] - Train Loss: {epoch_loss:.4f}")

# 损失曲线
plt.figure(figsize=(8, 5))
plt.plot(range(1, num_epochs + 1), train_losses)
plt.xlabel('Epoch')
plt.ylabel('Training Loss (MSE)')
plt.title('神经网络训练损失曲线')
plt.tight_layout()
plt.show()

# 测试集预测
model.eval()
with torch.no_grad():
    y_pred_nn = model(X_test_tensor.to(DEVICE)).cpu().numpy().flatten()

res_nn = evaluate_regression(y_test, y_pred_nn, "Neural Network (All Features)")
plot_prediction(y_test, y_pred_nn, "Neural Network (All Features)")

In [ ]:
# 10. 结果汇总

In [ ]:
results_df = pd.DataFrame([
    res_lr_all,
    res_lr_sel,
    res_lr_pca,
    res_nn,
]).sort_values(by='MSE', ascending=True).reset_index(drop=True)

print("\n" + "=" * 80)
print("7) 模型结果汇总（按 MSE 从小到大排序）")
print("=" * 80)
print(results_df)

# 柱状图比较各模型 MSE
plt.figure(figsize=(10, 5))
plt.bar(results_df['Model'], results_df['MSE'])
plt.xticks(rotation=20, ha='right')
plt.ylabel('MSE')
plt.title('不同模型测试集 MSE 对比')
plt.tight_layout()
plt.show()

In [ ]:
# 11. 简要结论（自动输出）

In [ ]:
best_model_name = results_df.loc[0, 'Model']
best_model_mse = results_df.loc[0, 'MSE']

print("\n" + "=" * 80)
print("8) 结论")
print("=" * 80)
print(f"在本次实验中，测试集上 MSE 最小的模型是：{best_model_name}")
print(f"其测试集 MSE 为：{best_model_mse:.4f}")
print("你可以在作业报告中结合相关性分析、PCA 结果和模型对比结果进行讨论。")